# PPO Metric Reosurce Managment

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

from environment.wrappers.time_limit_penalty import TimeLimitPenaltyWrapper
from environment.wrappers.action_flattener import FlattenActionWrapper
from environment.wrappers.zero_by_status import ZeroJobUsageByTheirStatus
from environment.core.jobs import JobStatus
from experiments.utils.wandb_custome_metric import CustomMetricsCallback

In [ ]:
from stable_baselines3.common.callbacks import CallbackList
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
import wandb
from wandb.integration.sb3 import WandbCallback
import logging

logging.basicConfig(level=logging.ERROR)

In [ ]:
total_timesteps = 500_000

policy_kwargs = dict(
    net_arch=[32,64,32]
)

config = {
    "policy_type": "MultiInputPolicy",
    "total_timesteps": total_timesteps,
    "env_name": "resource-management-online",
}

In [ ]:
run = wandb.init(
    project="cluster-scheduling",
    config=config,
    sync_tensorboard=True,
    monitor_gym=True,
    save_code=True,
)

In [ ]:
def make_env(
    n_jobs: int,
    n_machines: int,
    n_resources: int,
    n_ticks: int,
    max_episode_steps: int = 300,
    # not_termination_penalty: float = -1e4
):
    print(f"{n_jobs=}, {n_machines=}, {n_resources=}, {n_ticks=}, {max_episode_steps=}")
    env = gym.make(
        config["env_name"],
        render_mode="rgb_array",
        n_jobs=n_jobs,
        n_machines=n_machines,
        n_resources=n_resources,
        n_ticks=n_ticks,
        offline=True,
    )
    env = ZeroJobUsageByTheirStatus(env, JobStatus.Running, JobStatus.Completed, JobStatus.NotCreated)
    env = Monitor(env)
    return env

In [ ]:
env = DummyVecEnv([lambda: make_env(5, 1, 2, 5)])
env = VecVideoRecorder(
    env,
    f"videos/{run.id}",
    record_video_trigger=lambda x: x % 2_000 == 0,
    video_length=200,
)
model = PPO(
    config["policy_type"],
    env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    tensorboard_log=f"runs/{run.id}"
)
wandb_callback = WandbCallback(
    gradient_save_freq=1_000,
    model_save_path=f"models/{run.id}",
    verbose=2,
)
metric_callback = CustomMetricsCallback()
model.learn(
    total_timesteps=config["total_timesteps"],
    callback=CallbackList([
        metric_callback,
        wandb_callback
    ]),
)

In [ ]:
make_env(5, 3, 2, 10).observation_space, make_env(5, 3, 2, 10).action_space